# H3: ddddocr beta: помогает ли специализация на капчах?

**Проверяемая гипотеза:** Exact match выше, чем у PP-OCRv6 small.

Специализированный готовый OCR-SDK с ONNX-весами. Реальный граф модели анализируется отдельно; неизвестные детали обучения и корпуса не подменяются догадкой. Beta/common.onnx выбран заранее по профилю задачи; test не используется для выбора версии.

[Теория и исследования](../research/THEORY_AND_PAPERS.md). [Зафиксированный протокол](../research/PROTOCOL.md). Пороговые значения — наши инженерные критерии, не требования заказчика.

In [1]:
from pathlib import Path
import sys, json
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'captcha_lab').is_dir())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
import pandas as pd
from captcha_lab.evaluation import (manifest,run_local,metrics,scored,
                                      compare,all_summaries,hypothesis_results,RESULTS)
audit=json.loads((ROOT/'datasets/captcha-images/audit.json').read_text())
assert audit['holdout_images']==1984 and audit['shared_images']==200
display(pd.DataFrame([{'split':k,'n':v} for k,v in audit['source_split_counts'].items()]))
print('Holdout manifest SHA-256:',audit['holdout_manifest_sha256'])
print('Shared manifest SHA-256:',audit['shared_manifest_sha256'])
print('No prediction caches: inference cell below executes the actual model.')

,split,n
0,train,6000
1,validation,2000
2,test,2000


Holdout manifest SHA-256: 33fe043e8dec7b0e238c6bc81bb63c601699540b4b6c2025ad9480e21164f42b
Shared manifest SHA-256: 675bc9b4d93369e68817412fa08a72c6e21de23c820e00e42bc30d06c7390384
No prediction caches: inference cell below executes the actual model.


## Размеченная выборка

Публичный полный test содержит 2000 изображений; 16 просмотренных в предыдущем пилоте исключены. Метки и архивы скачаны полностью, не из preview. Функция распознавания получает только пиксели; метки присоединяются после предсказания. Исходные семь целевых меток выделены отдельно, два спорных примера исключены.

In [2]:
labels=manifest('shared_200' if 'ddddocr-beta'=='gemini' else 'holdout')
display(labels[['image_id','label','image_path']].head(12))
print('Evaluation images:',len(labels))

,image_id,label,image_path
0,test_000c1b6d4e5f11fb75d55a7c094b901e,161660,data/benchmark/images/test/161660.000c1b6d4e5f...
1,test_00105a4c053053509955d95368312987,496670,data/benchmark/images/test/496670.00105a4c0530...
2,test_001a3b40ef793b5984df8969f145d7bb,744782,data/benchmark/images/test/744782.001a3b40ef79...
3,test_0021e642491cf5bbc13cfc59fa46084a,017232,data/benchmark/images/test/017232.0021e642491c...
4,test_0047646a0e3b67ebc360e43ba88d1596,853091,data/benchmark/images/test/853091.0047646a0e3b...
5,test_0064b2188592a16991d5394b3ec5bf01,959686,data/benchmark/images/test/959686.0064b2188592...
6,test_00a5d81ab5060a0d45608169ce20bda9,159645,data/benchmark/images/test/159645.00a5d81ab506...
7,test_00e02c7c9da2cc0d3cdf5e3d9183a9f5,952376,data/benchmark/images/test/952376.00e02c7c9da2...
8,test_0123f4327a176c460aeefecd1d69bc11,374639,data/benchmark/images/test/374639.0123f4327a17...
9,test_0127972a796c1f306fdb5f919a7649fa,517493,data/benchmark/images/test/517493.0127972a796c...


Evaluation images: 1984


## Реальный прогон

Ячейка заново загружает модель и выполняет inference на каждой картинке. Ошибки сохраняются, готовые ответы не подставляются. CPU, batch=1, отдельный прогрев на validation.

In [3]:
frames=run_local('ddddocr-beta')

ddddocr-beta holdout 200 / 1984


ddddocr-beta holdout 400 / 1984


ddddocr-beta holdout 600 / 1984


ddddocr-beta holdout 800 / 1984


ddddocr-beta holdout 1000 / 1984


ddddocr-beta holdout 1200 / 1984


ddddocr-beta holdout 1400 / 1984


ddddocr-beta holdout 1600 / 1984


ddddocr-beta holdout 1800 / 1984


In [4]:
rows=[]
for dataset,predictions in frames.items():
    labels=manifest(dataset)
    rows.append(dict(dataset=dataset,**metrics(predictions,labels)))
    joined=scored(predictions,labels)
    print(dataset,'— ошибки по изображениям:')
    display(joined.loc[~joined.correct,['image_id','label','prediction','error','error_detail']].head(20))
display(pd.DataFrame(rows))

holdout — ошибки по изображениям:


,image_id,label,prediction,error,error_detail
26,test_03c01ef06d87fc2f119aeb5117da1566,499485,4eetca,,
28,test_03f34f710d4bbbcf8635367db15163a8,468785,463f,,
31,test_0430cbde3b764b430c4d3beb153149ef,468669,4f86fa,,
32,test_0434d7943346ab13e95cd4f382d89c0b,166294,ca94,,
33,test_043b263fc8724045aadac3391dc8147d,854970,8549,,
35,test_04adbfee6edb214e42c27172965ee67d,177753,17753,,
39,test_0525e75070bd757317867bcf2c363643,385269,85269,,
40,test_059a19ca75774a6ff857551a862a9eb9,772537,712537,,
41,test_059eb1c0e1f0a1b6037d5e7ffbc4bacc,241798,4179g,,
52,test_07225f814362d25e1be302032a81f982,461648,4616/r,,


target — ошибки по изображениям:


,image_id,label,prediction,error,error_detail
0,33ef91d464c03eb9,894412,894M27,,
1,41ada5001f1e1e88,432077,432oh,,
3,7619f1f1b372df6d,181263,1812E3,,
5,d4c0652824fb2dfd,639125,699f2,,
6,d4d1163a8ce99293,642044,6420M,,


,dataset,n,correct,exact_match,ci95_low,ci95_high,cer,errors,missing,median_ms,p95_ms,success_median_ms,success_p95_ms
0,holdout,1984,1465,0.738407,0.718621,0.757273,0.091230,0,0,18.95205,22.73078,18.95205,22.73078
1,target,7,2,0.285714,0.082219,0.641066,0.285714,0,0,32.57500,41.01147,32.57500,41.01147


## Вывод по гипотезе

Exact match и CER вычисляются по скачанным меткам. Вердикт получается из заранее заданного критерия и интервала разности; отсутствие статистической определённости явно показывается. Правильность на публичном наборе не переносится автоматически на Instagram.

In [5]:
summary=all_summaries()
display(summary[summary.model.eq('ddddocr-beta')])
hypotheses,paired=hypothesis_results()
display(hypotheses[hypotheses.hypothesis.eq('H3')])
display(paired[paired.hypothesis.eq('H3')] if not paired.empty else paired)

,model,dataset,n,correct,exact_match,ci95_low,ci95_high,cer,errors,missing,median_ms,p95_ms,success_median_ms,success_p95_ms
6,ddddocr-beta,holdout,1984,1465,0.738407,0.718621,0.757273,0.091230,0,0,18.95205,22.730780,18.95205,22.730780
7,ddddocr-beta,shared_200,200,152,0.760000,0.696267,0.813934,0.086667,0,0,18.98100,23.546005,18.98100,23.546005
8,ddddocr-beta,target,7,2,0.285714,0.082219,0.641066,0.285714,0,0,32.57500,41.011470,32.57500,41.011470


,hypothesis,criterion,verdict,estimate,ci_low,ci_high
2,H3,delta ddddocr-beta-ppocr-small>=0.00%,rejected,-0.207157,-0.23332,-0.179948


,hypothesis,model_a,model_b,n,wins,losses,delta,delta_ci95_low,delta_ci95_high,mcnemar_exact_p,holm_p
1,H3,ddddocr-beta,ppocr-small,1984,26,437,-0.207157,-0.23332,-0.179948,2.185392e-97,6.556177e-97
